### Dataset and Task Metadata

In [8]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="food_delivery_time",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/rajatkumar30/food-delivery-time",
    download_description="""
kaggle datasets download -d rajatkumar30/food-delivery-time -p local-data-warehouse/food_delivery_time/ --unzip
""",
    # References
    academic_reference_bibtex=r"""@misc{rajatkumar302023food,
  author       = {Kaggle User Rajatkumar30},
  title        = {Food Delivery Time},
  year         = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/rajatkumar30/food-delivery-time}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="rajatkumar302023food",
    license="Database Contents License (DbCL) v1.0",
    data_tags=["IID"],
    curation_comments="""
- We dropped entries with a duplicated ID, keeping only the first one.
- We dropped the ID column.
- Anomaly: The ID of the delivery person is given as a feature. In some contexts, this feature might not be allowed to use. Moreover, using this information might require considering a temporal split where a cold-start problem is covered.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Time_taken(min)",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [9]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/deliverytime.csv")

df = df.drop_duplicates(subset=["ID"])
df = df.drop(columns=["ID"])

cat_features = [
    "Delivery_person_ID",
    "Type_of_order",
    "Type_of_vehicle",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [10]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 45,451
Columns: 10
Use sampling: False (sample size: 45,451)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['Delivery_location_latitude', 'Delivery_location_longitude', 'Delivery_person_ID', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_person_Ratings', 'Delivery_person_Age', 'Type_of_order', 'Type_of_vehicle']
Rows remaining as candidates after top-9 filter: 414 (of 45,451)

#### Duplicate Report
Total duplicate rows: 12 (0.03% of dataset)
Duplicate rows ignoring target: 209 (0.46% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [11]:
# Sample Rows
df_head

,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Type_of_order,Type_of_vehicle,Time_taken(min)
0,AGRRES14DEL01,37,4.3,27.161850,78.040165,27.201850,78.080165,Snack,motorcycle,36
1,CHENRES13DEL03,29,5.0,13.027018,80.254791,13.117018,80.344791,Buffet,scooter,24
2,HYDRES14DEL02,22,4.1,17.426228,78.407495,17.506228,78.487495,Meal,motorcycle,34
3,COIMBRES06DEL03,38,4.6,11.021278,76.995017,11.051278,77.025017,Buffet,motorcycle,24
4,COIMBRES08DEL01,33,4.9,11.001852,76.976268,11.011852,76.986268,Meal,electric_scooter,28


In [12]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Delivery_person_ID,category,0.0,0.0,1320.0,"JAPRES11DEL02, JAPRES03DEL01, HYDRES04DEL02, PUNERES01DEL01, VADRES11DEL01, RANCHIRES02DEL01, BANGRES03DEL01, JAPRES09DEL02, VADRES14DEL01, VADRES11DEL02"
1,Type_of_order,category,0.0,0.0,4.0,"Snack , Meal , Drinks , Buffet"
2,Type_of_vehicle,category,0.0,0.0,4.0,"motorcycle , scooter , electric_scooter , bicycle"
3,Delivery_person_Ratings,float64,0.0,0.0,28.0,"4.6, 4.8, 4.7, 4.9, 5.0, 4.5, 4.1, 4.2, 4.3, 4.4"
4,Restaurant_latitude,float64,0.0,0.0,653.0,"0.0, 26.9114, 26.9141, 26.9029, 26.8923, 26.9029, 26.8884, 26.9137, 26.9053, 26.9023"
5,Restaurant_longitude,float64,0.0,0.0,515.0,"0.0, 75.8057, 75.789, 75.7929, 75.8069, 75.793, 75.8007, 75.7528, 75.7946, 75.8373"
6,Delivery_location_latitude,float64,0.0,0.0,4373.0,"0.13, 0.06, 0.09, 0.02, 0.07, 0.04, 0.05, 0.11, 0.01, 0.08"
7,Delivery_location_longitude,float64,0.0,0.0,4373.0,"0.13, 0.06, 0.09, 0.02, 0.07, 0.04, 0.05, 0.11, 0.01, 0.08"
8,Delivery_person_Age,int64,0.0,0.0,22.0,"29, 35, 36, 37, 30, 38, 24, 32, 22, 33"
9,Time_taken(min),int64,0.0,0.0,45.0,"26, 25, 27, 28, 29, 19, 15, 18, 16, 17"


In [13]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Delivery_person_Age,45451.0,29.542166,5.695052,15.000000,50.000000
Delivery_person_Ratings,45451.0,4.632360,0.327424,1.000000,6.000000
Restaurant_latitude,45451.0,17.022352,8.177316,-30.905562,30.914057
Restaurant_longitude,45451.0,70.237158,22.860620,-88.366217,88.433452
Delivery_location_latitude,45451.0,17.465516,7.336373,0.010000,31.054057
Delivery_location_longitude,45451.0,70.843062,21.122091,0.010000,88.563452
Time_taken(min),45451.0,26.290599,9.381744,10.000000,54.000000


In [14]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column             rank                                 
Delivery_person_ID 1         JAPRES11DEL02     67   0.15
                   2         JAPRES03DEL01     66   0.15
                   3         HYDRES04DEL02     66   0.15
                   4        PUNERES01DEL01     66   0.15
                   5         VADRES11DEL01     65   0.14
Type_of_order      1                Snack   11497  25.30
                   2                 Meal   11422  25.13
                   3               Drinks   11282  24.82
                   4               Buffet   11250  24.75
Type_of_vehicle    1           motorcycle   26366  58.01
                   2              scooter   15215  33.48
                   3     electric_scooter    3802   8.37
                   4              bicycle      68   0.15

In [15]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.486,-0.288,88.017,0.138,log,400374.3,424143.9,exponential


## Task Curation

In [16]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [17]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [18]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to food_delivery_time/019d5a56-f4f0-7a67-ba99-b4488a29da26
019d5a56-f4f0-7a67-ba99-b4488a29da26
2169f29d202affb78bc441e5bae67442523330fd945ae0b51558d9131551170f
